# Cell 1: Install zstd + Ollama + Cloudflare Tunnel (free, no signup needed)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Cell 2: Configuration — Choose your model!

# ─── Model presets ───────────────────────────────────────────
# Pick ONE based on your Colab GPU tier:
#
# T4 (free tier — 16GB VRAM):
#   'qwen2.5:7b'       → Fast, good quality, 5GB RAM, ~30 tok/s
#   'qwen2.5:14b'      → Slower but much better quality, 10GB RAM, ~15 tok/s
#   'llama3.1:8b'      → Meta's model, fast, 5GB RAM, ~35 tok/s
#   'mistral:7b'       → Mistral, fast, 5GB RAM, ~35 tok/s
#   'deepseek-r1:7b'   → Reasoning model, 5GB RAM, ~25 tok/s
#   'deepseek-r1:14b'  → Better reasoning, 10GB RAM, ~12 tok/s
#   'phi4:14b'         → Microsoft, strong reasoning, 10GB RAM, ~15 tok/s
#
# A100 (Colab Pro — 40GB VRAM):
#   'qwen2.5:32b'      → Excellent quality, 20GB RAM, ~20 tok/s
#   'llama3.1:70b'     → Premium quality, 40GB RAM, ~8 tok/s
#   'deepseek-r1:32b'  → Top reasoning, 20GB RAM, ~15 tok/s
#
# ─── Selection ───────────────────────────────────────────────

MODEL_NAME = 'qwen2.5:14b'  # ← Change this to switch models

# Auto-detect GPU type and warn if model too large
import subprocess
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode()
    gpu_name = gpu_info.split(',')[0].strip()
    vram_mb = int(gpu_info.split(',')[1].strip().replace(' MiB', ''))
    print(f'🎮 GPU: {gpu_name} ({vram_mb // 1024}GB VRAM)')
    
    # Rough VRAM requirements (quantized)
    MODEL_VRAM = {
        'qwen2.5:7b': 6, 'qwen2.5:14b': 11, 'qwen2.5:32b': 22,
        'llama3.1:8b': 6, 'llama3.1:70b': 42,
        'mistral:7b': 6, 'mixtral:8x7b': 26,
        'deepseek-r1:7b': 6, 'deepseek-r1:14b': 11, 'deepseek-r1:32b': 22,
        'phi4:14b': 11, 'gemma2:9b': 7, 'gemma2:27b': 18,
    }
    needed_gb = MODEL_VRAM.get(MODEL_NAME, 10)
    available_gb = vram_mb // 1024
    if needed_gb > available_gb:
        print(f'⚠️  WARNING: {MODEL_NAME} needs ~{needed_gb}GB VRAM but GPU has {available_gb}GB!')
        print(f'   Consider using a smaller model or upgrading to Colab Pro (A100).')
    else:
        print(f'✅ {MODEL_NAME} fits in {available_gb}GB VRAM (needs ~{needed_gb}GB)')
except Exception as e:
    print(f'⚠️ Could not detect GPU: {e}')

# ─── Other settings ──────────────────────────────────────────
# No ngrok token needed! Cloudflare Tunnel is 100% free.
REGENERATE_HOURS = 24  # Session regeneration interval

# Keep model warm in VRAM between requests (avoids reload delay)
KEEP_ALIVE = '30m'  # Ollama keep_alive: '5m', '30m', '1h', or '-1' (forever)

print(f'\n📋 Configuration:')
print(f'   Model: {MODEL_NAME}')
print(f'   Tunnel: Cloudflare (free, no signup)')
print(f'   Regeneration: every {REGENERATE_HOURS}h')
print(f'   Keep-alive: {KEEP_ALIVE}')

In [ ]:
# TODO: Delete this cell — duplicate of Cell 2 above (kept by notebook editor)
pass

In [ ]:
# Cell 4: Start Cloudflare Tunnel (FREE, no signup, no token needed)
import subprocess, time, re, os

# Kill any existing cloudflared
os.system('pkill cloudflared')
time.sleep(2)

# Start cloudflared tunnel on port 11434 (Ollama default)
# --url creates a quick tunnel with a random trycloudflare.com subdomain
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# Wait for the URL to appear in cloudflared output
OLLAMA_URL = None
for _ in range(30):
    line = proc.stdout.readline()
    if line:
        print(line.rstrip())
        # Cloudflare prints: "https://abc123.trycloudflare.com"
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            OLLAMA_URL = match.group(0)
            break
    time.sleep(1)

if OLLAMA_URL:
    print(f'\n? Ollama public URL: {OLLAMA_URL}')
    print(f'   → Copy this URL and paste it in the bot\'s .env:')
    print(f'   LOCAL_LLM_URL={OLLAMA_URL}/v1')
    print(f'   LOCAL_LLM_MODEL={MODEL_NAME},qwen2.5:7b-fast')
else:
    print('❌ Failed to get tunnel URL. Check cloudflared output above.')
    print('   You can also try: cloudflared tunnel --url http://localhost:11434')

In [ ]:
# Cell 5: Keep-alive + auto-regeneration loop
# This cell runs until Colab session expires, then you re-run the notebook

import time, requests, json, subprocess, re, os
from datetime import datetime, timedelta

start_time = datetime.now()
regenerate_at = start_time + timedelta(hours=REGENERATE_HOURS)

print(f'🕐 Session started at {start_time.strftime("%H:%M:%S")}')
print(f'🔄 Will regenerate at {regenerate_at.strftime("%H:%M:%S")} ({REGENERATE_HOURS}h)')
print(f'📊 Monitoring loop started (checks every 60s)...')
print(f'   Press Ctrl+C to stop manually')

health_ok_count = 0
health_fail_count = 0

try:
    while True:
        now = datetime.now()
        elapsed = now - start_time
        remaining = regenerate_at - now
        
        # Health check Ollama
        try:
            resp = requests.get(f'{OLLAMA_URL}/api/tags', timeout=10)
            if resp.status_code == 200:
                health_ok_count += 1
                status = '✅ healthy'
            else:
                health_fail_count += 1
                status = f'⚠️ status {resp.status_code}'
        except Exception as e:
            health_fail_count += 1
            status = f'❌ {str(e)[:50]}'
        
        # Print status every 5 minutes
        if int(elapsed.total_seconds()) % 300 == 0 and int(elapsed.total_seconds()) > 0:
            print(f'[{now.strftime("%H:%M:%S")}] uptime={int(elapsed.total_seconds()/60)}min '
                  f'remaining={int(remaining.total_seconds()/60)}min '
                  f'ok={health_ok_count} fail={health_fail_count} {status}')
        
        # Check if it's time to regenerate (restart tunnel)
        if now >= regenerate_at:
            print(f'🔄 Regeneration triggered at {now.strftime("%H:%M:%S")}')
            print('   Restarting Cloudflare tunnel...')
            
            # Kill old tunnel
            os.system('pkill cloudflared')
            time.sleep(3)
            
            # Start new tunnel
            new_proc = subprocess.Popen(
                ['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
            )
            for _ in range(30):
                line = new_proc.stdout.readline()
                if line:
                    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
                    if match:
                        OLLAMA_URL = match.group(0)
                        break
                time.sleep(1)
            print(f'   New URL: {OLLAMA_URL}')
            print(f'   ⚠️  Update the bot .env with the new URL!')
            
            # Reset timer
            regenerate_at = now + timedelta(hours=REGENERATE_HOURS)
            print(f'   Next regeneration at {regenerate_at.strftime("%H:%M:%S")}')
        
        time.sleep(60)
        
except KeyboardInterrupt:
    print('\n⏹️ Stopped by user')
except Exception as e:
    print(f'\n❌ Error: {e}')
finally:
    print(f'Session stats: ok={health_ok_count} fail={health_fail_count}')

In [ ]:
# TODO: Delete this cell — duplicate of Cell 5 above (kept by notebook editor)
pass

In [ ]:
# Cell 6 (optional): Test the LLM
import requests, json

test_url = f'{OLLAMA_URL}/v1/chat/completions'
payload = {
    'model': MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Bonjour, réponds en une phrase.'}],
    'max_tokens': 50,
    'stream': False
}

resp = requests.post(test_url, json=payload, timeout=60)
print(f'Status: {resp.status_code}')
if resp.status_code == 200:
    data = resp.json()
    print(f'Response: {data["choices"][0]["message"]["content"]}')
else:
    print(f'Error: {resp.text[:200]}')